# Naija-Speech — EDA → Curate to Hugging Face

**What this notebook does**, end to end, on Google Colab:

1. **Stage 1 — EDA (explore):** look at the *catalogue* of the Nigerian speech data
   (how many clips, hours, accents, speakers) **without downloading any audio**. Cheap.
2. **Stage 2 — Curate (port to HF):** stream every data source, reshape each into **one
   shared schema**, and upload the result as Parquet shards to **your private Hugging Face
   dataset** — disk-safe, so Colab never fills up.
3. **Stage 3 — Preview:** load the curated dataset back and eyeball the first **20 rows**
   (`.head(20)`) to confirm the text and tags look right.

> You are new to ML — every stage has a short plain-English note first. Skip the notes if you
> already know the idea.

**You need:** a Hugging Face account + a **write** token. A W&B key is optional here (only used
in later training steps).

## 0 — Get the code and install dependencies

We clone the repo and `pip install`. The one non-obvious pin is **`datasets<4.0`** — AfriSpeech-200
ships a *loading script*, and Hugging Face removed script-loading in `datasets` 4.x. The
`requirements.txt` already pins this.

In [ ]:
import os, subprocess

# Works on a fresh Colab (clones) or when already inside the repo.
if not os.path.exists("scripts/00_eda.py"):
    if not os.path.exists("naija-speech"):
        subprocess.run(
            ["git", "clone", "https://github.com/Johniblazee/naija-speech.git"],
            check=True,
        )
    os.chdir("naija-speech")

# Always pull latest so a RESUME session picks up pushed code changes.
subprocess.run(["git", "pull", "--ff-only"], check=False)
print("Working dir:", os.getcwd())

In [ ]:
# Install everything the pipeline needs (datasets<4.0 pin lives in requirements.txt).
!pip install -q -r requirements.txt

## 1 — Authenticate

- **Hugging Face token** (required): it uploads your curated dataset to *your* private HF storage,
  so it must have **write** access. Create one at
  <https://huggingface.co/settings/tokens>.
- **W&B key** (optional): only needed if you later log EDA/metrics to Weights & Biases. Press
  **Enter** to skip.

In [ ]:
import os, getpass
from huggingface_hub import login

# Hugging Face — needs WRITE access (it uploads your dataset).
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (write): ")
login(os.environ["HF_TOKEN"])

# Weights & Biases — optional. Press Enter to skip.
if not os.environ.get("WANDB_API_KEY"):
    wb = getpass.getpass("W&B API key (optional, Enter to skip): ")
    if wb:
        os.environ["WANDB_API_KEY"] = wb

## 2 — Stage 1: EDA (metadata only, no audio)

**Plain English:** EDA = *Exploratory Data Analysis*. Before ordering every book, you read the
library **catalogue**. Here we read the transcript CSVs (durations, accents, speakers, domains)
and skip the audio entirely — so it runs in seconds and costs no disk.

**Why it matters for the thesis:** it quantifies **data imbalance** (e.g. Hausa is thin), which
justifies choices later like *weighted sampling* and *adding more sources*. These numbers feed
Chapter 4.

In [ ]:
!python scripts/00_eda.py --config configs/data_afrispeech_ng.yaml --out outputs/eda

In [ ]:
# Show the generated report + charts inline.
from IPython.display import Markdown, Image, display
import glob, os

report = "outputs/eda/eda_report.md"
if os.path.exists(report):
    display(Markdown(open(report, encoding="utf-8").read()))

for png in sorted(glob.glob("outputs/eda/*.png")):
    display(Image(png))

## 3 — Stage 2: Curate all sources into ONE Hugging Face dataset

**The idea (plain English):** our data comes from **different datasets in different shapes**
(AfriSpeech-200 = short read clips; AfriSpeech-Dialog = long recorded conversations). We convert
every one into **a single shared table** so training can pull from all of them at once. Adding a
new dataset later = writing *one small adapter*, nothing else changes.

### The unified schema (the shared table's columns)

| Column | Meaning |
|---|---|
| `audio` | the waveform (kept at its **native** sample rate — see the resampling note below) |
| `text` / `text_raw` | normalized transcript / original transcript |
| `source` | which dataset it came from (`afrispeech-200`, `afrispeech-dialog`, …) |
| `language` | `en` / `en-codeswitch` / `pcm` (Nigerian Pidgin) / `yor` / `hau` / `ibo` |
| `task` | `stt` or `tts` |
| `accent`, `macro_accent` | raw accent, grouped into Yoruba / Igbo / Hausa / Other |
| `domain` | e.g. `clinical`, `conversational` |
| `speaker_id`, `gender`, `age_group` | speaker metadata |
| `duration` | seconds |
| `quality` | `unrated` / `clean` / `noisy` — used later to filter clean clips for **TTS** |
| `license` | so every clip carries its usage terms |

### Why streaming (and why your disk won't fill)

The old approach downloaded whole datasets **plus** a second local copy → Colab's disk filled and
nothing reached HF. Instead we **stream**: read a row → reshape it → buffer a few hundred → write
**one** Parquet shard → **upload it to HF** → **delete it locally**. Peak disk ≈ one shard.

### Note: audio resampling — *store native, resample late*

Whisper needs **16 kHz mono**, but we **store audio at its original rate** and only convert to
16 kHz **at training time**. Why: downsampling is **one-way and lossy** (it throws away everything
above 8 kHz). If we stored at 16 kHz, the later **TTS** track could never get the **24 kHz** it
wants for natural-sounding speech. So: keep the source faithful, resample late, per-task.

*(Full tradeoff table is in the chat / thesis §3. Short version: 16 kHz = model-compatible, 3×
smaller, keeps all speech intelligibility; the cost is discarded high frequencies that STT doesn't
need but TTS does.)*

### Note: the AfriSpeech-Dialog adapter

Its clips are long **multi-speaker conversations**; Whisper wants short single-speaker clips. The
adapter (`parse_dialog_turns`) reads the transcript's timestamps (`MM:SS:CC`) and `[Speaker N]:`
labels and **slices each conversation into per-turn utterances** — one short labelled clip per
speaker turn.

### Pick the run mode

The full Nigerian subset is large and **free Colab sessions time out / disconnect** — so the
pipeline **checkpoints every accent** and can resume across sessions.

- **First full run** → `FULL = True`, `RESUME = False`. Wipes the dataset and starts fresh,
  marking each accent done as it finishes.
- **After a disconnect** → `FULL = True`, `RESUME = True`, and re-run this section. It **skips
  accents already done** and continues — finished work is never re-downloaded. Repeat each new
  session until it completes.
- **Just validating?** → `FULL = False` runs a tiny smoke (~`MAX_PER_SPLIT` rows per source) in a
  minute.

> Expect the full port to span **several Colab sessions** — that's normal; resume makes it safe.
> To get the STT slice moving sooner, you can port only `[igbo, yoruba, hausa]` first (set
> `afrispeech_configs` in `configs/data_afrispeech_ng.yaml`), then expand later.

In [ ]:
FULL   = True    # True = port the ENTIRE Nigerian subset; False = tiny smoke run
RESUME = False   # True = CONTINUE an interrupted FULL run (skip done accents, no wipe)
MAX_PER_SPLIT = 20   # only used when FULL is False

mode = ("FULL port" if FULL else "SMOKE") + (" + RESUME" if (FULL and RESUME) else "")
print("Mode:", mode)

In [ ]:
# --push streams every source into your private HF dataset (disk-safe: peak disk ~ one shard).
cmd = "python scripts/01_build_corpus.py --push"
if not FULL:
    cmd += f" --max-per-split {MAX_PER_SPLIT}"   # smoke
elif RESUME:
    cmd += " --resume"                            # continue where the last session stopped

print("$", cmd, "\n")
!{cmd}

**What to look for in the output above:**
- one line **per accent**, e.g. `[curate] train/afrispeech-200/igbo: N clips in K shard(s)`,
  plus `[curate] train/afrispeech-dialog/dialog: N clips …`
- on a **resume** run, finished accents show `[resume] skip done: train/afrispeech-200/igbo`
- uploaded shards: `uploaded data/train-afrispeech-200-igbo-00000.parquet (…)`
- the final check: **`verify audio decodes on 'train': OK`**

If you see `[warn] could not list afrispeech configs … using cfg['accents']`, the network hiccuped
and it fell back to just 3 accents — re-run to get the full set.

## 4 — Stage 3: Preview the curated data (`.head(20)`)

Load the dataset back **by streaming** (so we don't re-download audio just to look at text/tags)
and show the first 20 rows. Note: a Hugging Face `Dataset` has **no** `.head()` — that's a pandas
method — so we take 20 rows and view them as a pandas `DataFrame`.

In [ ]:
from datasets import load_dataset, Audio
import pandas as pd

REPO = "Johniblazee/naija-speech-afrispeech-ng"  # = hf_curated_repo in configs/data_afrispeech_ng.yaml

# Stream + don't decode audio (decode=False) — we only want to eyeball text + tags cheaply.
stream = (load_dataset(REPO, split="train", streaming=True)
          .cast_column("audio", Audio(decode=False)))

rows = list(stream.take(20))
df = pd.DataFrame(rows).drop(columns=["audio"])   # drop the raw bytes from the preview
df.head(20)

### (Optional) Listen to one clip — at the 16 kHz STT training rate

This is the *only* place we actually decode + resample audio, to hear what the STT model will get.

In [ ]:
from datasets import load_dataset, Audio
import IPython.display as ipd

ex = next(iter(
    load_dataset(REPO, split="train", streaming=True)
    .cast_column("audio", Audio(sampling_rate=16000))   # decode + resample to 16 kHz
))
print(f"[{ex['source']} | {ex['macro_accent']} | {ex['domain']}] {ex['text'][:80]}")
ipd.Audio(ex["audio"]["array"], rate=ex["audio"]["sampling_rate"])

## What's next

- Add the **Common Voice (NG)** adapter (one more `*_stream` generator in `src/curate.py`).
- Then the **STT vertical slice**: zero-shot Whisper baseline → LoRA fine-tune (Unsloth) →
  accent-stratified WER → W&B. That produces the first real Chapter 4/5 numbers.

Progress is logged in `docs/PROGRESS.md`.